Benefits of Prompt Chaining with LangGraph
Improved Context Management: By breaking tasks into smaller prompts, the model can focus on one aspect at a time, reducing the risk of losing context in long inputs.
Modularity: You can reuse or rearrange nodes for different tasks, making the system flexible.
Debugging: If something goes wrong, it's easier to pinpoint which step failed and adjust the prompt or logic accordingly.
Complex Reasoning: Chaining prompts allows the model to "think" step-by-step, mimicking human problem-solving more effectively.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq

#os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="qwen-2.5-32b")
#llm = ChatOpenAI(model="gpt-4o")

result = llm.invoke("Hello")
result

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display


## Graph State
class State(TypedDict):
    topic: str
    story: str
    improved_story: str
    final_story: str


## Nodes

## Nodes

def generate_story(state: State):
    msg = llm.invoke(
        f"Write a one sentence story premise about {state['topic']}"
    )
    return {"story": msg.content}


def check_conflict(state: State):
    if "?" in state["story"] or "!" in state["story"]:
        return "Fail"

    return "Pass"

def improved_story(state: State):
    msg = llm.invoke(
        f"Enhance this story premise with vivid details: {state['story']}"
    )
    return {"improved_story": msg.content}


def polish_story(state: State):
    msg = llm.invoke(
        f"Add an unexpected twist to this story premise: {state['improved_story']}"
    )
    return {"final_story": msg.content}

In [ ]:
graph = StateGraph(State)

graph.add_node("generate", generate_story)
graph.add_node("improve", improved_story)
graph.add_node("polish", polish_story)


## Define the edges

graph.add_edge(START, "generate")

graph.add_conditional_edges(
    "generate",
    check_conflict,
    {
        "Pass": "improve",
        "Fail": "generate"
    }
)

graph.add_edge("improve", "polish")
graph.add_edge("polish", END)


# Compile the graph
compiled_graph = graph.compile()


# Visualize the graph (for Jupyter notebook)
graph_image = compiled_graph.get_graph().draw_mermaid_png()
display(Image(graph_image))

In [ ]:
state = {
    "topic": "Agentic AI Systems"
}

result = compiled_graph.invoke(state)

result